In [ ]:
import json
from pathlib import Path
from typing import Optional, List, Tuple
import pandas as pd
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
import os
print(os.getcwd())

In [ ]:
def collect_data(directory: Path):
    records = []

    for file in tqdm(directory.glob('*')):
        with open(file) as f:
            data = json.load(f)
            if 'error' in data['result']:
                continue
            args = pd.Series(data['args'])
            result = pd.Series(data['result'])
            record = pd.concat([args, result])
            records.append(record)

    return pd.DataFrame(records)

def __collapse(s):
    return "profs" if s in ["BandCamp", "beets_test", "bandcamp-free"] else s

In [ ]:
# get median sections
all_probes = collect_data(Path("../../generated_data/datagen.tasks.task_ffprobe"))

def _calc(duration: float) -> Tuple[float, Optional[float]]:
    if duration < 32:
        return 0, None
    else:
        return duration / 2, 30

starts_durations = all_probes.duration.map(_calc)
tracks_with_sections = all_probes.copy()
tracks_with_sections["start"] = [s for s,d in starts_durations]
tracks_with_sections["duration"] = [d for s,d in starts_durations]
display(tracks_with_sections)

tracks_with_sections.to_csv("../track_lists/all_x.csv")


In [ ]:
def sample_balanced(
    df: pd.DataFrame,
    class_col: str,
    include_classes: Optional[List[str]] = None,
    random_state: Optional[int] = None,
) -> pd.DataFrame:
    # None means include all classes. Otherwise, select just those in the list.
    if include_classes is not None:
        df = df.loc[df[class_col].isin(include_classes)]
    g = df.groupby(class_col)
    return g.apply(lambda group: group.sample(g.size().min(), random_state=random_state)).reset_index(level=0)


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn import svm
from sklearn.neural_network import MLPClassifier
from sklearn import preprocessing

def experiment(df: pd.DataFrame, X_col: str, y_col: str = 'dataset'):
    X = np.stack(df[X_col].to_list())
    X = preprocessing.StandardScaler().fit_transform(X)
    y = pd.Categorical(df[y_col]).codes
    clf = svm.SVC(kernel='linear', C=1)
    # clf = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=0, max_iter=1000)
    scores = cross_validate(clf, X, y, scoring=['precision', 'recall', 'f1'])
    return scores

fip_scores = experiment(balanced_fip, 'fip')
spf_scores = experiment(balanced_spf, 'spf')

print(fip_scores)
print(spf_scores)

In [ ]:
from sklearn.cluster import KMeans

X = np.stack(fip_df['fip'].to_list())
km = KMeans(n_clusters=8)
km.fit(X)
cluster_centres = np.argmin(km.transform(X), axis=0)

fip_cluster_df = fip_df.iloc[cluster_centres]
display(fip_cluster_df)
for f in fip_cluster_df.filepath:
    display(Audio(f), f)

fig, axes = plt.subplots(2,4, figsize=(8,4), sharex=True, sharey=True)
axes = axes.flatten()
for ax, (_, row) in zip(axes, fip_cluster_df.iterrows()):
    ax.plot(row.fip)
    ax.set_ylim(-60, 1)

plt.show()

In [ ]:
rfdf = collect_data(Path("../../generated_data/datagen.tasks.task_recording_features"))
louddf = collect_data(Path("../../generated_data/datagen.tasks.task_loudness"))
spfdf = collect_data(Path("../../generated_data/datagen.tasks.task_spf"))
fipdf = collect_data(Path("../../generated_data/datagen.tasks.task_fip"))
df = rfdf.copy().set_index("filepath")
df = df.join(louddf.set_index("filepath").drop(["start", "duration"], axis=1))
df = df.join(spfdf.set_index("filepath").drop(["start", "duration"], axis=1))
df = df.join(fipdf.set_index("filepath").drop(["start", "duration"], axis=1))
df["dataset"] = df.index.map(lambda p: __collapse(Path(p).parts[4]))
display(df)

In [ ]:
plt.hist(np.log(df.stereo_width + 0.0001), bins=50)